#Data Preprocessing:

Load the dataset and separate the input features (Outlook, Temperature, Humidity, Wind) from the target variable (Play).

Convert all categorical feature values and target labels into numerical representations using an appropriate encoding technique.


In [ ]:
import pandas as pd
data=pd.read_csv("lab8.csv")

In [ ]:
data.head()

,No,Outlook,Temperature,Humidity,Wind,Play Tennis
0,1,Sunny,Hot,High,Weak,No
1,2,Sunny,Hot,High,Strong,No
2,3,Overcast,Hot,High,Weak,Yes
3,4,Rain,Mild,High,Weak,Yes
4,5,Rain,Cool,Normal,Weak,Yes


In [ ]:
data.describe()

,No
count,50.00000
mean,25.50000
std,14.57738
min,1.00000
25%,13.25000
50%,25.50000
75%,37.75000
max,50.00000


In [ ]:
data.columns

Index(['No', 'Outlook', 'Temperature', 'Humidity', 'Wind', 'Play Tennis'], dtype='object')

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   No           50 non-null     int64 
 1   Outlook      50 non-null     object
 2   Temperature  50 non-null     object
 3   Humidity     50 non-null     object
 4   Wind         50 non-null     object
 5   Play Tennis  50 non-null     object
dtypes: int64(1), object(5)
memory usage: 2.5+ KB


In [ ]:
data.shape

(50, 6)

In [ ]:
from sklearn.preprocessing import LabelEncoder


categorical_cols = ['Outlook', 'Temperature', 'Humidity', 'Wind', 'Play Tennis']


data_encoded = data.copy()

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    data_encoded[col] = le.fit_transform(data_encoded[col])
    label_encoders[col] = le

print("DataFrame after encoding categorical features and target variable:")
print(data_encoded.head())

DataFrame after encoding categorical features and target variable:
   No  Outlook  Temperature  Humidity  Wind  Play Tennis
0   1        2            1         0     1            0
1   2        2            1         0     0            0
2   3        0            1         0     1            1
3   4        1            2         0     1            1
4   5        1            0         1     1            1


#Dataset Partitioning:
Divide the dataset into training and testing subsets using an 80:20 train-test split ratio


In [ ]:
from sklearn.model_selection import train_test_split


X = data_encoded.drop(['No', 'Play Tennis'], axis=1)
y = data_encoded['Play Tennis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (40, 4)
Shape of X_test: (10, 4)
Shape of y_train: (40,)
Shape of y_test: (10,)


#Naive Bayes Model Training & Evaluation:

Train a Categorical Naive Bayes (CategoricalNB) model on the training data

Predict the class labels for the test dataset

Calculate and display the overall Model Accuracy

Display the Confusion Matrix and Classification Report (Precision, Recall, F1-Score).


In [ ]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


cnb = CategoricalNB()
cnb.fit(X_train, y_train)


y_pred = cnb.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}\n")


conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)
print("\n")


class_report = classification_report(y_test, y_pred)
print("Classification Report:")
print(class_report)

Model Accuracy: 0.8000

Confusion Matrix:
[[1 1]
 [1 7]]


Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.88      0.88      0.88         8

    accuracy                           0.80        10
   macro avg       0.69      0.69      0.69        10
weighted avg       0.80      0.80      0.80        10



#Single-Sample Inference:

Predict whether a person will play tennis under the following specific weather conditions:

Outlook: Sunny

Temperature: Cool

Humidity: High

Wind: Strong

Display both the predicted class label and the corresponding class probabilities


In [ ]:

new_sample_features = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

encoded_sample = {}
for feature, value in new_sample_features.items():

    if feature in label_encoders:
        encoded_sample[feature] = label_encoders[feature].transform([value])[0]
    else:
        print(f"Warning: Encoder not found for feature '{feature}'")


new_sample_df = pd.DataFrame([encoded_sample], columns=X_train.columns)


predicted_label_encoded = cnb.predict(new_sample_df)[0]


predicted_proba = cnb.predict_proba(new_sample_df)[0]


predicted_label_decoded = label_encoders['Play Tennis'].inverse_transform([predicted_label_encoded])[0]


print(f"New Sample Conditions: {new_sample_features}")
print(f"Predicted Class Label: {predicted_label_decoded}")
print(f"Class Probabilities: {predicted_proba}")


class_names = label_encoders['Play Tennis'].classes_
proba_dict = dict(zip(class_names, predicted_proba))
print(f"Class Probabilities (Decoded): {proba_dict}")

New Sample Conditions: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted Class Label: No
Class Probabilities: [0.92560203 0.07439797]
Class Probabilities (Decoded): {'No': np.float64(0.9256020301956863), 'Yes': np.float64(0.07439796980431378)}


#Model Comparison:
Train a Decision Tree Classifier and a Logistic Regression Classifier, and an SVM on the same training data.
Compare all three models in terms of test accuracy, single-sample query prediction, and output class probabilities.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

print("\n--- Decision Tree Classifier ---")
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)


y_pred_dt = dt_classifier.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print(f"Test Accuracy: {accuracy_dt:.4f}")


predicted_label_dt_encoded = dt_classifier.predict(new_sample_df)[0]
predicted_proba_dt = dt_classifier.predict_proba(new_sample_df)[0]
predicted_label_dt_decoded = label_encoders['Play Tennis'].inverse_transform([predicted_label_dt_encoded])[0]

print(f"Single Sample Predicted Class Label: {predicted_label_dt_decoded}")
proba_dict_dt = dict(zip(class_names, predicted_proba_dt))
print(f"Single Sample Class Probabilities: {proba_dict_dt}")


print("\n--- Logistic Regression Classifier ---")

log_reg_classifier = LogisticRegression(random_state=42, solver='liblinear')
log_reg_classifier.fit(X_train, y_train)


y_pred_lr = log_reg_classifier.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Test Accuracy: {accuracy_lr:.4f}")


predicted_label_lr_encoded = log_reg_classifier.predict(new_sample_df)[0]
predicted_proba_lr = log_reg_classifier.predict_proba(new_sample_df)[0]
predicted_label_lr_decoded = label_encoders['Play Tennis'].inverse_transform([predicted_label_lr_encoded])[0]

print(f"Single Sample Predicted Class Label: {predicted_label_lr_decoded}")
proba_dict_lr = dict(zip(class_names, predicted_proba_lr))
print(f"Single Sample Class Probabilities: {proba_dict_lr}")


print("\n--- SVM Classifier ---")

svm_classifier = SVC(random_state=42, probability=True)
svm_classifier.fit(X_train, y_train)


y_pred_svm = svm_classifier.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"Test Accuracy: {accuracy_svm:.4f}")

predicted_label_svm_encoded = svm_classifier.predict(new_sample_df)[0]
predicted_proba_svm = svm_classifier.predict_proba(new_sample_df)[0]
predicted_label_svm_decoded = label_encoders['Play Tennis'].inverse_transform([predicted_label_svm_encoded])[0]

print(f"Single Sample Predicted Class Label: {predicted_label_svm_decoded}")
proba_dict_svm = dict(zip(class_names, predicted_proba_svm))
print(f"Single Sample Class Probabilities: {proba_dict_svm}")


--- Decision Tree Classifier ---
Test Accuracy: 0.8000
Single Sample Predicted Class Label: No
Single Sample Class Probabilities: {'No': np.float64(1.0), 'Yes': np.float64(0.0)}

--- Logistic Regression Classifier ---
Test Accuracy: 0.4000
Single Sample Predicted Class Label: No
Single Sample Class Probabilities: {'No': np.float64(0.9466476048933914), 'Yes': np.float64(0.05335239510660861)}

--- SVM Classifier ---
Test Accuracy: 0.7000
Single Sample Predicted Class Label: No
Single Sample Class Probabilities: {'No': np.float64(0.9751413656922103), 'Yes': np.float64(0.024858634307789732)}


naive bayes algo and decision tree perfrm better on multi class claasification with 80% accuracy

svm classifier with accuracy 70%  which is also suitable for this type of classification peoblem

logistic regression give the least accuracy of 40%